# 쇼핑몰 상품 추천 RAG v1  
## 임베딩 + 거래데이터 + 날짜 리드타임 기반 추천

v1의 목표는 v0처럼 단순 TF-IDF나 수동 의도 사전에 의존하지 않고,  
**거래데이터의 구매 맥락과 날짜, 현재 상품데이터의 의미 유사도**를 함께 사용해 추천하는 것입니다.

---

## v0와 달라진 점

| 구분 | v0 | v1 |
|---|---|---|
| 상품 검색 | TF-IDF 중심 | Ollama 임베딩 중심, 실패 시 TF-IDF fallback |
| 의도 확장 | 수동 사전 가능 | 수동 상품 사전 사용 안 함 |
| 거래데이터 | 약한 힌트 | 추천 방향의 핵심 신호 |
| 날짜 | 거의 미반영 | 행사/사용월 기준 1~2개월 앞선 주문월 반영 |
| 추천 점수 | 상품 텍스트 유사도 중심 | 거래 맥락 + 상품 임베딩 + 날짜 + 예산/수량 |
| AI 사용 | 선택적 답변 생성 | 질문 구조화 + 추천 설명 생성 가능 |

---

## 핵심 전제

판촉물은 보통 행사 당월에 주문하는 것이 아니라,  
**사용/배포 예정일보다 1~2개월 앞서 주문**하는 경우가 많습니다.

예:

```text
8월 행사/배포용
→ 거래데이터에서는 6~7월 주문 데이터가 더 중요
→ 8월 주문 데이터는 보조 참고
```

따라서 거래데이터의 날짜는 아래처럼 해석합니다.

```text
거래데이터 날짜 = 주문일자
사용자 질문의 월/계절 = 사용 또는 배포 예정 시점
```


## 0. 설치

처음 한 번만 실행합니다.

Ollama 임베딩을 쓰려면 아래가 필요합니다.

```bash
python -m pip install ollama
ollama pull bge-m3
```

한국어 추천 설명까지 생성하려면:

```bash
ollama pull gemma3
```

또는:

```bash
ollama pull qwen3
```

> `bge-m3`가 없거나 `ollama` 패키지가 없으면 자동으로 TF-IDF fallback으로 동작하게 구성했습니다.


In [ ]:
# 필요 시 한 번만 실행
# !pip install pandas openpyxl scikit-learn numpy ollama


## 1. 기본 설정 및 파일 경로

In [ ]:
from pathlib import Path
import re
import html
import json
import math
import hashlib
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

# =========================
# 파일 경로
# =========================
PRODUCT_PATH = Path("../data/상품데이터_샘플100.xlsx")
TRADE_PATH = Path("../data/거래데이터_샘플100.xlsx")

# ChatGPT 샌드박스 경로에서 실행하는 경우
if not PRODUCT_PATH.exists():
    PRODUCT_PATH = Path("../data/상품데이터_샘플100.xlsx")
if not TRADE_PATH.exists():
    TRADE_PATH = Path("../data/거래데이터_샘플100.xlsx")

OUTPUT_DIR = Path("../output")
CACHE_DIR = Path("../cache")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

# =========================
# 임베딩 설정
# =========================
USE_OLLAMA_EMBEDDING = True
EMBED_MODEL = "bge-m3"

# LLM 조건 추출/답변 생성용 모델
LLM_MODEL = "gemma3:4b"

# 상품 추천 결과 개수
DEFAULT_TOP_K = 5

print("PRODUCT_PATH:", PRODUCT_PATH)
print("TRADE_PATH:", TRADE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## 2. 파일 로딩 및 컬럼 확인

In [ ]:
# =========================
# 엑셀 로딩
# =========================

product_raw = pd.read_excel(PRODUCT_PATH, dtype=str).fillna("")
trade_raw = pd.read_excel(TRADE_PATH, dtype=str).fillna("")

print("상품데이터 shape:", product_raw.shape)
print("거래데이터 shape:", trade_raw.shape)

print("\n[상품데이터 컬럼]")
for i, col in enumerate(product_raw.columns, start=1):
    print(f"{i:02d}. {col}")

print("\n[거래데이터 컬럼]")
for i, col in enumerate(trade_raw.columns, start=1):
    print(f"{i:02d}. {col}")

display(product_raw.head(3))
display(trade_raw.head(3))


## 3. 공통 전처리 함수

In [ ]:
# =========================
# 공통 전처리 함수
# =========================

def clean_text(x):
    if pd.isna(x):
        return ""

    x = str(x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = html.unescape(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def to_number(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s in ["", "-", "nan", "None", "null"]:
        return np.nan

    s = re.sub(r"[^0-9.]", "", s)

    if s == "":
        return np.nan

    try:
        return float(s)
    except Exception:
        return np.nan


def safe_col(df, col):
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)


def combine_text(*parts):
    return clean_text(" ".join([str(p) for p in parts if str(p).strip()]))


def normalize_vector_matrix(mat: np.ndarray) -> np.ndarray:
    mat = np.array(mat, dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return mat / norms


def cosine_scores(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    query_vec = np.array(query_vec, dtype=np.float32)
    if query_vec.ndim == 1:
        query_vec = query_vec.reshape(1, -1)

    query_norm = np.linalg.norm(query_vec, axis=1, keepdims=True)
    query_norm[query_norm == 0] = 1
    query_vec = query_vec / query_norm

    return (matrix @ query_vec.T).ravel()


def text_hash(texts):
    joined = "\n".join([str(x) for x in texts])
    return hashlib.md5(joined.encode("utf-8")).hexdigest()[:10]


## 4. 상품데이터 정규화

현재 추천 가능한 상품 후보 DB를 만듭니다.

상품 검색용 텍스트에는 아래를 포함합니다.

```text
상품명
브랜드
모델명
대/중/소 카테고리
검색키워드
간략한설명
상품내용
제조사/원산지
```


In [ ]:
# =========================
# 상품데이터 정규화
# =========================

product_df = product_raw.copy()

product_df["product_id"] = safe_col(product_df, "상품번호")
product_df.loc[product_df["product_id"].str.strip() == "", "product_id"] = safe_col(product_df, "상품코드")

product_df["brand"] = safe_col(product_df, "브랜드").map(clean_text)
product_df["product_name"] = safe_col(product_df, "상품명").map(clean_text)
product_df["model_name"] = safe_col(product_df, "모델명").map(clean_text)

product_df["price"] = safe_col(product_df, "상품판매가").map(to_number)
product_df["supply_price"] = safe_col(product_df, "상품공급가").map(to_number)
product_df["moq"] = safe_col(product_df, "최소구매수량").map(to_number)

product_df["category_large"] = safe_col(product_df, "대 카테고리").map(clean_text)
product_df["category_middle"] = safe_col(product_df, "중 카테고리").map(clean_text)
product_df["category_small"] = safe_col(product_df, "소 카테고리").map(clean_text)
product_df["category_detail"] = safe_col(product_df, "세분류").map(clean_text)

product_df["category_path"] = (
    product_df["category_large"] + " > " +
    product_df["category_middle"] + " > " +
    product_df["category_small"] + " > " +
    product_df["category_detail"]
).map(clean_text)

product_df["keywords"] = safe_col(product_df, "검색키워드").map(clean_text)
product_df["summary"] = safe_col(product_df, "간략한설명").map(clean_text)
product_df["detail_text"] = safe_col(product_df, "상품내용").map(clean_text)
product_df["manufacturer"] = safe_col(product_df, "제조사").map(clean_text)
product_df["origin"] = safe_col(product_df, "원산지").map(clean_text)
product_df["image"] = safe_col(product_df, "큰이미지").map(clean_text)

product_df["member_only"] = safe_col(product_df, "회원전용상품").map(clean_text)
product_df["stock"] = safe_col(product_df, "재고").map(to_number)

product_df["product_search_text"] = (
    "[상품명] " + product_df["product_name"] + "\n" +
    "[브랜드] " + product_df["brand"] + "\n" +
    "[모델명] " + product_df["model_name"] + "\n" +
    "[카테고리] " + product_df["category_path"] + "\n" +
    "[검색키워드] " + product_df["keywords"] + "\n" +
    "[설명] " + product_df["summary"] + " " + product_df["detail_text"] + "\n" +
    "[제조/원산지] " + product_df["manufacturer"] + " " + product_df["origin"]
).map(clean_text)

product_df = product_df[product_df["product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 상품 수:", len(product_df))
display(product_df[[
    "product_id", "product_name", "price", "moq", "category_path", "keywords"
]].head(10))


## 5. 거래데이터 정규화

거래데이터는 직접 추천 상품 DB가 아니라,  
**과거 구매 맥락과 상품군 패턴을 찾는 데이터**로 사용합니다.

거래 검색용 텍스트에는 아래를 포함합니다.

```text
구매처 분류
구매처명
주문월
상품명
상품분류
행사별/대상별/시즌별 필터
인쇄방법
가격대
최소구매수량
```


In [ ]:
# =========================
# 거래데이터 정규화
# =========================

trade_df = trade_raw.copy()

buyer_name_col = "구매처 명 "
if buyer_name_col not in trade_df.columns and "구매처 명" in trade_df.columns:
    buyer_name_col = "구매처 명"

trade_df["buyer_type_large"] = safe_col(trade_df, "구매처 분류(대)").map(clean_text)
trade_df["buyer_type_middle"] = safe_col(trade_df, "구매처 분류(중)").map(clean_text)
trade_df["buyer_type_small"] = safe_col(trade_df, "구매처 분류(소)").map(clean_text)
trade_df["buyer_type_detail"] = safe_col(trade_df, "구매처 분류(세)").map(clean_text)
trade_df["buyer_name"] = safe_col(trade_df, buyer_name_col).map(clean_text)

trade_df["buyer_type_path"] = (
    trade_df["buyer_type_large"] + " > " +
    trade_df["buyer_type_middle"] + " > " +
    trade_df["buyer_type_small"] + " > " +
    trade_df["buyer_type_detail"]
).map(clean_text)

trade_df["date_raw"] = safe_col(trade_df, "날짜").map(clean_text)
trade_df["order_date"] = pd.to_datetime(trade_df["date_raw"], errors="coerce")
trade_df["order_year"] = trade_df["order_date"].dt.year
trade_df["order_month"] = trade_df["order_date"].dt.month

trade_df["trade_product_name"] = safe_col(trade_df, "상품").map(clean_text)

trade_df["trade_category_large"] = safe_col(trade_df, "상품분류(대)").map(clean_text)
trade_df["trade_category_middle"] = safe_col(trade_df, "상품분류(중)").map(clean_text)
trade_df["trade_category_small"] = safe_col(trade_df, "상품분류(소)").map(clean_text)

trade_df["trade_category_path"] = (
    trade_df["trade_category_large"] + " > " +
    trade_df["trade_category_middle"] + " > " +
    trade_df["trade_category_small"]
).map(clean_text)

trade_df["bulk_price"] = safe_col(trade_df, "대량가격(원)").map(to_number)
trade_df["middle_price"] = safe_col(trade_df, "중간가격(원)").map(to_number)
trade_df["small_price"] = safe_col(trade_df, "소량가격(원)").map(to_number)
trade_df["trade_moq"] = safe_col(trade_df, "최소구매수량").map(to_number)

trade_df["event_filter"] = safe_col(trade_df, "행사별(필터)").map(clean_text)
trade_df["target_filter"] = safe_col(trade_df, "대상별(필터)").map(clean_text)
trade_df["season_filter"] = safe_col(trade_df, "시즌별(필터)").map(clean_text)
trade_df["print_method"] = safe_col(trade_df, "인쇄 방법").map(clean_text)
trade_df["memo"] = safe_col(trade_df, "비  고").map(clean_text)

trade_df["trade_search_text"] = (
    "[구매처] " + trade_df["buyer_type_path"] + " " + trade_df["buyer_name"] + "\n" +
    "[주문월] " + trade_df["order_month"].fillna("").astype(str) + "월\n" +
    "[상품] " + trade_df["trade_product_name"] + "\n" +
    "[상품분류] " + trade_df["trade_category_path"] + "\n" +
    "[행사/대상/시즌] " + trade_df["event_filter"] + " " + trade_df["target_filter"] + " " + trade_df["season_filter"] + "\n" +
    "[인쇄] " + trade_df["print_method"] + "\n" +
    "[비고] " + trade_df["memo"]
).map(clean_text)

trade_df = trade_df[trade_df["trade_product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 거래 수:", len(trade_df))
display(trade_df[[
    "date_raw", "order_month", "buyer_type_path", "buyer_name",
    "trade_product_name", "trade_category_path",
    "bulk_price", "middle_price", "small_price",
    "trade_moq", "event_filter", "target_filter", "season_filter"
]].head(10))


## 6. 질문 조건 추출

v1에서는 두 가지 방식이 있습니다.

1. **Ollama LLM 조건 추출**  
   - 구매처/행사/목적/계절/월/예산/수량 등을 JSON으로 추출
2. **정규식 fallback**  
   - Ollama가 없거나 실패할 때 예산/수량/월/계절만 간단히 추출

수동 상품 사전은 사용하지 않습니다.  
단, 계절의 월 범위는 달력 개념이므로 사용합니다.


In [ ]:
# =========================
# 질문 조건 추출
# =========================

SEASON_EVENT_MONTHS = {
    "봄": [3, 4, 5],
    "여름": [6, 7, 8],
    "가을": [9, 10, 11],
    "겨울": [12, 1, 2],
}


def extract_basic_conditions(query: str) -> dict:
    q = str(query)

    budget = None

    # 예: 3000원, 3,000원
    m = re.search(r"(\d+(?:,\d+)*)\s*원\s*(?:이하|미만|안쪽|대로)?", q)
    if m:
        budget = int(m.group(1).replace(",", ""))

    # 예: 3천원, 3천 원, 3천원대
    if budget is None:
        m = re.search(r"(\d+)\s*천\s*원(?:대)?", q)
        if m:
            budget = int(m.group(1)) * 1000

    # 예: 5만원, 5만 원
    if budget is None:
        m = re.search(r"(\d+)\s*만\s*원", q)
        if m:
            budget = int(m.group(1)) * 10000

    quantity = None

    # 예: 500개, 1,000개, 500EA
    m = re.search(r"(\d+(?:,\d+)*)\s*(?:개|ea|EA|pcs|PCS)", q)
    if m:
        quantity = int(m.group(1).replace(",", ""))

    event_month = None

    # 예: 8월, 8월에, 8월달
    m = re.search(r"(1[0-2]|[1-9])\s*월", q)
    if m:
        event_month = int(m.group(1))

    season = None
    for s in SEASON_EVENT_MONTHS:
        if s in q:
            season = s
            break

    return {
        "raw_query": q,
        "buyer_context": "",
        "event_context": "",
        "purpose": "",
        "season": season,
        "event_month": event_month,
        "budget_max": budget,
        "quantity": quantity,
        "style_keywords": [],
        "must_have": [],
        "avoid": [],
        "extraction_method": "regex_fallback",
    }


def safe_json_loads(text: str) -> dict:
    text = str(text).strip()

    # 코드블록 제거
    text = re.sub(r"^```json", "", text).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    # JSON 객체 부분만 추출
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end >= 0 and end > start:
        text = text[start:end+1]

    return json.loads(text)


def extract_query_profile_with_ollama(query: str, model: str = LLM_MODEL) -> dict:
    """
    Ollama LLM으로 질문을 구조화합니다.
    실패하면 정규식 fallback 결과를 반환합니다.
    """

    fallback = extract_basic_conditions(query)

    try:
        import ollama
    except ModuleNotFoundError:
        return fallback

    system_prompt = """
너는 판촉물 상품 추천을 위한 조건 추출기다.
사용자 질문에서 추천에 필요한 조건만 JSON으로 추출한다.

주의:
- 상품 추천 사전을 만들지 않는다.
- "여름이면 선풍기"처럼 상품군을 임의 확장하지 않는다.
- 사용자가 말한 구매처, 행사, 목적, 계절, 월, 예산, 수량, 스타일만 구조화한다.
- 모르는 값은 null 또는 빈 문자열/빈 리스트로 둔다.
- 반드시 JSON만 출력한다.
"""

    user_prompt = f"""
아래 사용자 요청을 JSON으로 구조화해줘.

[사용자 요청]
{query}

[출력 JSON 스키마]
{{
  "buyer_context": "구매처/업종/기관/대상 예: 병원, 대학교, 기업, 어린이집",
  "event_context": "행사/상황 예: 개원, 창립기념, 박람회, 여름행사",
  "purpose": "용도 예: 답례품, 사은품, 홍보물, 기념품",
  "season": "봄/여름/가을/겨울 중 하나 또는 null",
  "event_month": 1~12 숫자 또는 null,
  "budget_max": 숫자 또는 null,
  "quantity": 숫자 또는 null,
  "style_keywords": ["실용적", "고급", "저렴한"] 형태,
  "must_have": ["로고인쇄", "휴대성"] 형태,
  "avoid": []
}}
"""

    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": user_prompt.strip()},
            ],
            options={"temperature": 0.0, "num_ctx": 2048},
            stream=False,
        )

        content = response["message"]["content"]
        parsed = safe_json_loads(content)

        result = fallback.copy()
        for k, v in parsed.items():
            if k in result:
                result[k] = v

        # 정규식에서 잡힌 예산/수량/월이 있는데 LLM이 놓치면 보존
        basic = extract_basic_conditions(query)
        for k in ["budget_max", "quantity", "event_month", "season"]:
            if result.get(k) in [None, "", []] and basic.get(k) not in [None, "", []]:
                result[k] = basic[k]

        result["raw_query"] = query
        result["extraction_method"] = "ollama_llm"
        return result

    except Exception as e:
        fallback["extraction_error"] = str(e)
        return fallback


# 테스트
for q in [
    "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
    "병원 개원 답례품으로 3천원 이하 500개 추천해줘",
    "대학교 OT에서 나눠줄 저렴한 사은품 추천해줘",
]:
    print("\nQUERY:", q)
    print(extract_query_profile_with_ollama(q))


## 7. 행사/사용월 기준 참고 주문월 계산

판촉물은 사용월보다 1~2개월 앞서 주문되는 경우가 많으므로,  
사용자가 말한 월을 그대로 거래데이터 주문월에 대입하지 않습니다.

예:

```text
8월 행사
→ 7월 주문: 강한 가중치
→ 6월 주문: 강한 가중치
→ 8월 주문: 보조 가중치
```


In [ ]:
# =========================
# 날짜/리드타임 점수
# =========================

def prev_month(month: int, n: int = 1) -> int:
    m = month - n
    while m <= 0:
        m += 12
    return m


def get_reference_order_month_weights(event_month: int) -> dict:
    """
    사용/행사월 기준으로 거래데이터 주문월 가중치를 반환합니다.

    event_month=8이면:
    7월: 1.0
    6월: 0.85
    8월: 0.45
    5월: 0.25
    """
    if event_month is None or pd.isna(event_month):
        return {}

    event_month = int(event_month)

    weights = {
        prev_month(event_month, 1): 1.00,
        prev_month(event_month, 2): 0.85,
        event_month: 0.45,
        prev_month(event_month, 3): 0.25,
    }

    return weights


def get_event_months_from_profile(profile: dict) -> list:
    event_month = profile.get("event_month")
    season = profile.get("season")

    months = []

    if event_month not in [None, "", np.nan]:
        try:
            months.append(int(event_month))
        except Exception:
            pass

    if season in SEASON_EVENT_MONTHS:
        months.extend(SEASON_EVENT_MONTHS[season])

    # 중복 제거
    return sorted(set(months))


def get_reference_month_weights_from_profile(profile: dict) -> dict:
    event_months = get_event_months_from_profile(profile)

    combined = {}

    for m in event_months:
        w = get_reference_order_month_weights(m)
        for month, score in w.items():
            combined[month] = max(combined.get(month, 0), score)

    return combined


def calc_date_lead_score(order_month, profile: dict) -> float:
    if pd.isna(order_month):
        return 0.0

    weights = get_reference_month_weights_from_profile(profile)

    if not weights:
        return 0.0

    try:
        order_month = int(order_month)
    except Exception:
        return 0.0

    return float(weights.get(order_month, 0.0))


# 테스트
for q in ["8월 행사 판촉물", "여름 행사 판촉물", "12월 연말 선물"]:
    profile = extract_basic_conditions(q)
    print(q)
    print("profile:", profile)
    print("event_months:", get_event_months_from_profile(profile))
    print("reference_order_month_weights:", get_reference_month_weights_from_profile(profile))
    print()


## 8. 임베딩 생성

기본은 Ollama `bge-m3` 임베딩입니다.

실패하면 자동으로 TF-IDF fallback을 사용합니다.

### 권장 흐름

```bash
ollama pull bge-m3
python -m pip install ollama
```


In [ ]:
# =========================
# 임베딩 생성
# =========================

def try_ollama_embed_texts(texts, model=EMBED_MODEL, batch_size=32):
    """
    Ollama Python 패키지 버전 차이를 고려해
    ollama.embed()와 ollama.embeddings()를 모두 지원합니다.
    """
    import ollama

    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = [str(x) for x in texts[start:start+batch_size]]

        try:
            # 최신 API: ollama.embed(model=..., input=[...])
            resp = ollama.embed(model=model, input=batch)
            if "embeddings" in resp:
                all_embeddings.extend(resp["embeddings"])
                continue
        except Exception:
            pass

        # fallback: 구버전/단건 API
        for text in batch:
            try:
                resp = ollama.embeddings(model=model, prompt=text)
                all_embeddings.append(resp["embedding"])
            except Exception as e:
                raise RuntimeError(f"Ollama embedding 실패: {e}")

    return np.array(all_embeddings, dtype=np.float32)


def build_embedding_or_tfidf(name, texts, use_ollama=True, model=EMBED_MODEL):
    """
    임베딩 또는 TF-IDF 인덱스를 생성합니다.
    Ollama 임베딩 성공 시:
        backend='ollama', matrix=정규화 벡터
    실패 시:
        backend='tfidf', vectorizer, matrix=sparse tfidf
    """

    texts = [str(x) for x in texts]

    cache_key = text_hash(texts)
    cache_path = CACHE_DIR / f"{name}_{model}_{cache_key}.npy"

    if use_ollama:
        try:
            if cache_path.exists():
                matrix = np.load(cache_path)
                matrix = normalize_vector_matrix(matrix)
                print(f"[{name}] Ollama embedding cache 로딩:", cache_path)
                return {
                    "backend": "ollama",
                    "model": model,
                    "matrix": matrix,
                    "vectorizer": None,
                    "cache_path": cache_path,
                }

            print(f"[{name}] Ollama embedding 생성 중... model={model}, rows={len(texts)}")
            matrix = try_ollama_embed_texts(texts, model=model)
            np.save(cache_path, matrix)
            matrix = normalize_vector_matrix(matrix)

            print(f"[{name}] Ollama embedding 저장:", cache_path)

            return {
                "backend": "ollama",
                "model": model,
                "matrix": matrix,
                "vectorizer": None,
                "cache_path": cache_path,
            }

        except Exception as e:
            warnings.warn(f"[{name}] Ollama embedding 실패. TF-IDF로 fallback합니다. 원인: {e}")

    print(f"[{name}] TF-IDF fallback 생성")
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=1,
    )

    matrix = vectorizer.fit_transform(texts)

    return {
        "backend": "tfidf",
        "model": "tfidf_char_ngram",
        "matrix": matrix,
        "vectorizer": vectorizer,
        "cache_path": None,
    }


def search_index(query_text, index, top_k=None):
    """
    Ollama embedding 또는 TF-IDF 인덱스에서 검색합니다.
    반환값: score array
    """

    if index["backend"] == "ollama":
        q_vec = try_ollama_embed_texts([query_text], model=index["model"])
        q_vec = normalize_vector_matrix(q_vec)
        scores = cosine_scores(q_vec[0], index["matrix"])
        return scores

    if index["backend"] == "tfidf":
        q_vec = index["vectorizer"].transform([query_text])
        scores = cosine_similarity(q_vec, index["matrix"]).flatten()
        return scores

    raise ValueError(f"지원하지 않는 backend: {index['backend']}")


product_index = build_embedding_or_tfidf(
    name="product",
    texts=product_df["product_search_text"].tolist(),
    use_ollama=USE_OLLAMA_EMBEDDING,
    model=EMBED_MODEL,
)

trade_index = build_embedding_or_tfidf(
    name="trade",
    texts=trade_df["trade_search_text"].tolist(),
    use_ollama=USE_OLLAMA_EMBEDDING,
    model=EMBED_MODEL,
)

print("product backend:", product_index["backend"])
print("trade backend:", trade_index["backend"])


## 9. 유사 거래 사례 검색

사용자 질문과 비슷한 거래를 찾을 때, 단순 의미 유사도만 보지 않습니다.

```text
trade_total_score =
거래 임베딩 유사도
+ 날짜 리드타임 점수
+ 예산/수량 유사도
+ 시즌 필터 보조 점수
```

즉, “8월 행사”라는 질문은 8월 주문보다 6~7월 주문 데이터에 더 높은 점수를 줄 수 있습니다.


In [ ]:
# =========================
# 유사 거래 사례 검색
# =========================

def build_trade_query_text(profile: dict) -> str:
    parts = [
        profile.get("raw_query", ""),
        profile.get("buyer_context", ""),
        profile.get("event_context", ""),
        profile.get("purpose", ""),
        profile.get("season", ""),
        " ".join(profile.get("style_keywords") or []),
        " ".join(profile.get("must_have") or []),
    ]
    return combine_text(*parts)


def calc_trade_condition_score(row, profile: dict) -> float:
    score = 0.0

    budget = profile.get("budget_max")
    quantity = profile.get("quantity")
    season = profile.get("season")

    # 예산과 거래 가격대 유사성
    if budget not in [None, ""] and not pd.isna(budget):
        try:
            budget = float(budget)
            prices = [
                row.get("bulk_price", np.nan),
                row.get("middle_price", np.nan),
                row.get("small_price", np.nan),
            ]
            valid_prices = [p for p in prices if not pd.isna(p)]

            if valid_prices:
                min_price = min(valid_prices)
                if min_price <= budget:
                    score += 0.35
                elif min_price <= budget * 1.2:
                    score += 0.15
        except Exception:
            pass

    # 수량과 최소구매수량
    if quantity not in [None, ""] and not pd.isna(quantity):
        try:
            quantity = float(quantity)
            moq = row.get("trade_moq", np.nan)
            if pd.isna(moq) or moq <= quantity:
                score += 0.25
        except Exception:
            pass

    # 시즌 필터가 명시되어 있는 경우
    if season and season in str(row.get("season_filter", "")):
        score += 0.20

    return min(score, 1.0)


def retrieve_similar_trades(query: str, top_k: int = 20, profile: dict | None = None) -> pd.DataFrame:
    if profile is None:
        profile = extract_query_profile_with_ollama(query)

    trade_query_text = build_trade_query_text(profile)

    semantic_scores = search_index(trade_query_text, trade_index)

    result = trade_df.copy()
    result["trade_semantic_score"] = semantic_scores
    result["date_lead_score"] = result["order_month"].apply(lambda m: calc_date_lead_score(m, profile))
    result["trade_condition_score"] = result.apply(lambda row: calc_trade_condition_score(row, profile), axis=1)

    # 가중치:
    # 의미 유사도 55%
    # 날짜 리드타임 25%
    # 예산/수량/시즌 보조 20%
    result["trade_total_score"] = (
        result["trade_semantic_score"] * 0.55 +
        result["date_lead_score"] * 0.25 +
        result["trade_condition_score"] * 0.20
    )

    return (
        result
        .sort_values("trade_total_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"
test_profile = extract_query_profile_with_ollama(test_query)

print("[추출 프로필]")
print(json.dumps(test_profile, ensure_ascii=False, indent=2))

print("\n[참고 주문월 가중치]")
print(get_reference_month_weights_from_profile(test_profile))

similar_trades = retrieve_similar_trades(test_query, top_k=10, profile=test_profile)

display(similar_trades[[
    "trade_total_score", "trade_semantic_score", "date_lead_score", "trade_condition_score",
    "date_raw", "order_month", "buyer_type_path", "buyer_name",
    "trade_product_name", "trade_category_path",
    "event_filter", "target_filter", "season_filter"
]])


## 10. 유사 거래 사례에서 추천 신호 추출

여기서 중요한 점은 **거래 상품명과 현재 상품명을 1:1 매칭하지 않는 것**입니다.

대신 유사 거래 사례에서 아래 신호를 추출합니다.

```text
자주 등장한 상품분류
자주 등장한 거래 상품명
가격대
최소구매수량
구매처/행사 맥락
시즌 필터
```

이 신호를 하나의 텍스트로 만들어 현재 상품데이터와 임베딩 유사도를 계산합니다.


In [ ]:
# =========================
# 거래 신호 추출
# =========================

def weighted_value_counts(df, col, weight_col="trade_total_score", top_n=10):
    rows = []

    for _, row in df.iterrows():
        value = clean_text(row.get(col, ""))
        if not value:
            continue
        weight = float(row.get(weight_col, 0))
        rows.append((value, weight))

    if not rows:
        return pd.DataFrame(columns=[col, "weighted_score", "count"])

    temp = pd.DataFrame(rows, columns=[col, "weight"])
    out = (
        temp.groupby(col)
        .agg(weighted_score=("weight", "sum"), count=("weight", "count"))
        .reset_index()
        .sort_values("weighted_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return out


def extract_trade_signals(similar_trades: pd.DataFrame) -> dict:
    category_large = weighted_value_counts(similar_trades, "trade_category_large", top_n=10)
    category_middle = weighted_value_counts(similar_trades, "trade_category_middle", top_n=10)
    category_small = weighted_value_counts(similar_trades, "trade_category_small", top_n=10)
    products = weighted_value_counts(similar_trades, "trade_product_name", top_n=10)
    buyers = weighted_value_counts(similar_trades, "buyer_type_middle", top_n=10)
    events = weighted_value_counts(similar_trades, "event_filter", top_n=10)
    targets = weighted_value_counts(similar_trades, "target_filter", top_n=10)
    seasons = weighted_value_counts(similar_trades, "season_filter", top_n=10)

    # 가격대 통계
    price_cols = ["bulk_price", "middle_price", "small_price"]
    prices = []

    for c in price_cols:
        if c in similar_trades.columns:
            prices.extend([
                x for x in similar_trades[c].tolist()
                if not pd.isna(x)
            ])

    price_stats = {}
    if prices:
        arr = np.array(prices, dtype=float)
        price_stats = {
            "min": float(np.nanmin(arr)),
            "q25": float(np.nanpercentile(arr, 25)),
            "median": float(np.nanmedian(arr)),
            "q75": float(np.nanpercentile(arr, 75)),
            "max": float(np.nanmax(arr)),
        }

    # 신호 텍스트 구성
    signal_parts = []

    def add_top_values(title, df, col):
        vals = df[col].head(8).tolist() if len(df) else []
        if vals:
            signal_parts.append(f"[{title}] " + " ".join(vals))

    add_top_values("유사거래 상품분류 대", category_large, "trade_category_large")
    add_top_values("유사거래 상품분류 중", category_middle, "trade_category_middle")
    add_top_values("유사거래 상품분류 소", category_small, "trade_category_small")
    add_top_values("유사거래 상품명", products, "trade_product_name")
    add_top_values("유사구매처", buyers, "buyer_type_middle")
    add_top_values("행사", events, "event_filter")
    add_top_values("대상", targets, "target_filter")
    add_top_values("시즌", seasons, "season_filter")

    signal_text = clean_text("\n".join(signal_parts))

    return {
        "category_large": category_large,
        "category_middle": category_middle,
        "category_small": category_small,
        "products": products,
        "buyers": buyers,
        "events": events,
        "targets": targets,
        "seasons": seasons,
        "price_stats": price_stats,
        "signal_text": signal_text,
    }


signals = extract_trade_signals(similar_trades)

print("[거래 신호 텍스트]")
print(signals["signal_text"])

print("\n[가격대 통계]")
print(signals["price_stats"])

print("\n[상위 상품분류 중]")
display(signals["category_middle"])

print("\n[상위 거래 상품명]")
display(signals["products"])


## 11. 상품 후보 랭킹

v1의 상품 점수는 아래 기준으로 계산합니다.

```text
final_score =
상품-질문 임베딩 유사도
+ 상품-거래신호 임베딩 유사도
+ 거래데이터 기반 상품분류 신호 점수
+ 예산 점수
+ 수량 점수
+ 상품 데이터 품질 점수
```

### 점수 비중

| 점수 | 비중 | 의미 |
|---|---:|---|
| query_product_score | 20% | 사용자 질문과 현재 상품의 의미 유사도 |
| trade_signal_product_score | 30% | 유사 거래 신호와 현재 상품의 의미 유사도 |
| trade_category_signal_score | 20% | 유사 거래에서 나온 상품분류와 현재 상품분류의 일치 |
| budget_score | 15% | 예산 조건 충족 |
| quantity_score | 5% | 최소구매수량 조건 충족 |
| product_quality_score | 10% | 가격/상품명/카테고리 등 추천에 필요한 정보 보유 |

이 구조는 TF-IDF보다 **거래 맥락과 실제 상품군 패턴**을 더 중요하게 봅니다.


In [ ]:
# =========================
# 상품 랭킹
# =========================

def calc_budget_score(price, budget):
    if budget in [None, ""] or pd.isna(budget):
        return 0.5

    if pd.isna(price):
        return 0.2

    try:
        price = float(price)
        budget = float(budget)

        if price <= budget:
            return 1.0
        if price <= budget * 1.2:
            return 0.65
        if price <= budget * 1.5:
            return 0.35
        return 0.0
    except Exception:
        return 0.2


def calc_quantity_score(moq, quantity):
    if quantity in [None, ""] or pd.isna(quantity):
        return 0.5

    if pd.isna(moq):
        return 0.7

    try:
        moq = float(moq)
        quantity = float(quantity)

        if moq <= quantity:
            return 1.0
        if moq <= quantity * 1.5:
            return 0.5
        return 0.0
    except Exception:
        return 0.5


def calc_product_quality_score(row):
    score = 0.0

    if clean_text(row.get("product_name", "")):
        score += 0.25
    if clean_text(row.get("category_path", "")):
        score += 0.25
    if not pd.isna(row.get("price", np.nan)):
        score += 0.25
    if clean_text(row.get("keywords", "")) or clean_text(row.get("summary", "")):
        score += 0.25

    return score


def calc_trade_category_signal_score(row, signals: dict):
    """
    거래데이터에서 자동 추출된 상품분류 신호와
    현재 상품의 카테고리가 겹치는지 봅니다.

    수동 상품 사전이 아니라,
    유사 거래 사례에서 나온 분류를 그대로 사용합니다.
    """

    product_text = combine_text(
        row.get("category_path", ""),
        row.get("product_name", ""),
        row.get("keywords", ""),
    )

    score = 0.0

    # 상품분류 중/소를 중심으로 봄
    for signal_key, col_name, weight in [
        ("category_middle", "trade_category_middle", 0.55),
        ("category_small", "trade_category_small", 0.35),
        ("category_large", "trade_category_large", 0.10),
    ]:
        signal_df = signals.get(signal_key)

        if signal_df is None or len(signal_df) == 0:
            continue

        max_weighted = signal_df["weighted_score"].max()

        for _, srow in signal_df.iterrows():
            value = clean_text(srow.get(col_name, ""))
            if not value:
                continue

            if value in product_text:
                normalized = float(srow["weighted_score"]) / max_weighted if max_weighted else 0
                score += normalized * weight

    return min(score, 1.0)


def build_product_query_text(profile: dict) -> str:
    parts = [
        profile.get("raw_query", ""),
        profile.get("buyer_context", ""),
        profile.get("event_context", ""),
        profile.get("purpose", ""),
        profile.get("season", ""),
        " ".join(profile.get("style_keywords") or []),
        " ".join(profile.get("must_have") or []),
    ]
    return combine_text(*parts)


def rank_products(query: str, profile: dict, signals: dict, top_k: int = 5) -> pd.DataFrame:
    product_query_text = build_product_query_text(profile)

    # 1) 사용자 질문과 상품의 의미 유사도
    query_product_scores = search_index(product_query_text, product_index)

    # 2) 거래 신호와 상품의 의미 유사도
    signal_text = signals.get("signal_text", "")

    if signal_text:
        trade_signal_product_scores = search_index(signal_text, product_index)
    else:
        trade_signal_product_scores = np.zeros(len(product_df))

    result = product_df.copy()

    result["query_product_score"] = query_product_scores
    result["trade_signal_product_score"] = trade_signal_product_scores

    # 3) 거래데이터 기반 상품분류 신호 점수
    result["trade_category_signal_score"] = result.apply(
        lambda row: calc_trade_category_signal_score(row, signals),
        axis=1,
    )

    # 4) 예산/수량/품질 점수
    budget = profile.get("budget_max")
    quantity = profile.get("quantity")

    result["budget_score"] = result["price"].apply(lambda p: calc_budget_score(p, budget))
    result["quantity_score"] = result["moq"].apply(lambda q: calc_quantity_score(q, quantity))
    result["product_quality_score"] = result.apply(calc_product_quality_score, axis=1)

    # 5) 최종 점수
    result["final_score"] = (
        result["query_product_score"] * 0.20 +
        result["trade_signal_product_score"] * 0.30 +
        result["trade_category_signal_score"] * 0.20 +
        result["budget_score"] * 0.15 +
        result["quantity_score"] * 0.05 +
        result["product_quality_score"] * 0.10
    )

    # 6) 추천 사유
    def make_reason(row):
        reasons = []

        if row["trade_signal_product_score"] >= 0.55:
            reasons.append("유사 거래 사례에서 나타난 상품군과 의미적으로 가까움")
        elif row["trade_signal_product_score"] >= 0.35:
            reasons.append("유사 거래 신호와 일부 관련 있음")

        if row["trade_category_signal_score"] >= 0.5:
            reasons.append("과거 유사 거래에서 나온 상품분류와 현재 상품분류가 일치")
        elif row["trade_category_signal_score"] > 0:
            reasons.append("과거 유사 거래 상품분류와 일부 겹침")

        if row["query_product_score"] >= 0.55:
            reasons.append("사용자 요청과 상품 설명의 의미 유사도 높음")
        elif row["query_product_score"] >= 0.35:
            reasons.append("사용자 요청과 상품 설명이 일부 유사함")

        if budget not in [None, ""] and not pd.isna(budget):
            if pd.notna(row["price"]) and float(row["price"]) <= float(budget):
                reasons.append(f"예산 {int(float(budget)):,}원 이하 조건 충족")
            elif pd.notna(row["price"]):
                reasons.append("예산 초과 여부 확인 필요")
            else:
                reasons.append("가격 정보 확인 필요")

        if quantity not in [None, ""] and not pd.isna(quantity):
            if pd.isna(row["moq"]) or float(row["moq"]) <= float(quantity):
                reasons.append("요청 수량 기준 최소구매수량 충족 가능")
            else:
                reasons.append("최소구매수량 확인 필요")

        if not reasons:
            reasons.append("현재 상품데이터 기준 추천 후보로 검토 가능")

        return " / ".join(reasons)

    result["recommend_reason"] = result.apply(make_reason, axis=1)

    output_cols = [
        "product_id",
        "product_name",
        "price",
        "moq",
        "category_path",
        "keywords",
        "query_product_score",
        "trade_signal_product_score",
        "trade_category_signal_score",
        "budget_score",
        "quantity_score",
        "product_quality_score",
        "final_score",
        "recommend_reason",
        "image",
    ]

    return (
        result
        .sort_values("final_score", ascending=False)
        .head(top_k)
        [output_cols]
        .reset_index(drop=True)
    )


## 12. 최종 추천 함수

이 함수 하나로 전체 흐름을 실행합니다.

```text
사용자 질문
↓
AI/정규식 조건 추출
↓
참고 주문월 계산
↓
유사 거래 검색
↓
거래 신호 추출
↓
현재 상품 랭킹
```


In [ ]:
# =========================
# 최종 추천 함수
# =========================

def recommend_products_v1(query: str, top_k: int = DEFAULT_TOP_K, trade_top_k: int = 30):
    profile = extract_query_profile_with_ollama(query)

    similar_trades = retrieve_similar_trades(
        query=query,
        top_k=trade_top_k,
        profile=profile,
    )

    signals = extract_trade_signals(similar_trades)

    recommendations = rank_products(
        query=query,
        profile=profile,
        signals=signals,
        top_k=top_k,
    )

    return {
        "query": query,
        "profile": profile,
        "reference_month_weights": get_reference_month_weights_from_profile(profile),
        "similar_trades": similar_trades,
        "signals": signals,
        "recommendations": recommendations,
        "backend": {
            "product": product_index["backend"],
            "trade": trade_index["backend"],
            "embedding_model": product_index.get("model"),
        }
    }


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"
result = recommend_products_v1(test_query, top_k=5, trade_top_k=20)

print("[질문]")
print(result["query"])

print("\n[추출 프로필]")
print(json.dumps(result["profile"], ensure_ascii=False, indent=2))

print("\n[참고 주문월 가중치]")
print(result["reference_month_weights"])

print("\n[사용 backend]")
print(result["backend"])

print("\n[유사 거래 신호 텍스트]")
print(result["signals"]["signal_text"])

print("\n[추천 상품]")
display(result["recommendations"])

print("\n[유사 거래 사례]")
display(result["similar_trades"][[
    "trade_total_score", "trade_semantic_score", "date_lead_score", "trade_condition_score",
    "date_raw", "order_month", "buyer_type_path", "buyer_name",
    "trade_product_name", "trade_category_path",
    "event_filter", "target_filter", "season_filter"
]].head(10))


## 13. 여러 질문 일괄 테스트 및 결과 저장

In [ ]:
# =========================
# 여러 질문 테스트
# =========================

test_queries = [
    "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
    "병원 개원 답례품으로 3천원 이하 500개 추천해줘",
    "대학교 OT에서 나눠줄 저렴한 사은품 추천해줘",
    "회사 창립기념품으로 실용적인 상품 추천해줘",
    "박람회 부스에서 나눠줄 홍보물 추천해줘",
    "12월 연말 고객 선물로 5만원 이하 상품 추천해줘",
    "어린이집 행사 답례품 추천해줘",
    "로고 인쇄 가능한 텀블러 추천해줘",
]

all_rows = []

for query in test_queries:
    result = recommend_products_v1(query, top_k=5, trade_top_k=20)

    profile = result["profile"]
    ref_months = result["reference_month_weights"]

    top_trade_categories = []
    if len(result["signals"]["category_middle"]):
        top_trade_categories = result["signals"]["category_middle"]["trade_category_middle"].head(5).tolist()

    for rank, (_, row) in enumerate(result["recommendations"].iterrows(), start=1):
        all_rows.append({
            "query": query,
            "rank": rank,
            "buyer_context": profile.get("buyer_context", ""),
            "event_context": profile.get("event_context", ""),
            "purpose": profile.get("purpose", ""),
            "season": profile.get("season", ""),
            "event_month": profile.get("event_month", ""),
            "reference_month_weights": json.dumps(ref_months, ensure_ascii=False),
            "top_trade_categories": ", ".join(top_trade_categories),
            "product_id": row["product_id"],
            "product_name": row["product_name"],
            "price": row["price"],
            "moq": row["moq"],
            "category_path": row["category_path"],
            "query_product_score": row["query_product_score"],
            "trade_signal_product_score": row["trade_signal_product_score"],
            "trade_category_signal_score": row["trade_category_signal_score"],
            "budget_score": row["budget_score"],
            "quantity_score": row["quantity_score"],
            "product_quality_score": row["product_quality_score"],
            "final_score": row["final_score"],
            "recommend_reason": row["recommend_reason"],
        })

batch_result_df = pd.DataFrame(all_rows)

BATCH_RESULT_PATH = OUTPUT_DIR / "reco_v1_embedding_tradedate_batchtest.xlsx"
batch_result_df.to_excel(BATCH_RESULT_PATH, index=False)

print("추천 결과 저장:", BATCH_RESULT_PATH)
display(batch_result_df.head(30))


## 14. Ollama로 고객용 추천 답변 생성

랭킹과 점수 계산은 코드가 하고,  
최종 문장화만 LLM이 하도록 분리합니다.

중요 규칙:

```text
- 추천 후보와 유사 거래 신호 안에 있는 내용만 사용
- 가격/수량/납기 등 데이터에 없는 내용은 단정하지 않기
- 추천 이유는 점수 항목을 기반으로 작성
```


In [ ]:
# =========================
# Ollama 추천 답변 생성
# =========================

def build_answer_context(result: dict) -> str:
    lines = []

    lines.append("[질문]")
    lines.append(result["query"])
    lines.append("")

    lines.append("[추출 프로필]")
    lines.append(json.dumps(result["profile"], ensure_ascii=False))
    lines.append("")

    lines.append("[참고 주문월 가중치]")
    lines.append(json.dumps(result["reference_month_weights"], ensure_ascii=False))
    lines.append("")

    lines.append("[거래데이터 신호]")
    lines.append(result["signals"].get("signal_text", ""))
    lines.append("")

    lines.append("[추천 상품 후보]")
    for i, (_, row) in enumerate(result["recommendations"].iterrows(), start=1):
        lines.append(f"{i}. {row['product_name']}")
        lines.append(f"   - 가격: {row['price']}")
        lines.append(f"   - 최소구매수량: {row['moq']}")
        lines.append(f"   - 카테고리: {row['category_path']}")
        lines.append(f"   - 최종점수: {row['final_score']:.4f}")
        lines.append(f"   - 추천근거: {row['recommend_reason']}")
        lines.append("")

    lines.append("[유사 거래 사례 Top 5]")
    for i, (_, row) in enumerate(result["similar_trades"].head(5).iterrows(), start=1):
        lines.append(f"{i}. {row['trade_product_name']}")
        lines.append(f"   - 날짜: {row['date_raw']}")
        lines.append(f"   - 구매처: {row['buyer_type_path']} / {row['buyer_name']}")
        lines.append(f"   - 상품분류: {row['trade_category_path']}")
        lines.append(f"   - 점수: {row['trade_total_score']:.4f}")
        lines.append("")

    return "\n".join(lines)


def generate_customer_answer_with_ollama(query: str, model: str = LLM_MODEL, top_k: int = 5):
    try:
        import ollama
    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "ollama 패키지가 없습니다. 현재 가상환경에서 `python -m pip install ollama`를 실행하세요."
        )

    result = recommend_products_v1(query, top_k=top_k, trade_top_k=30)
    context = build_answer_context(result)

    system_prompt = """
너는 쇼핑몰 판촉물 상품 추천 어시스턴트다.

규칙:
- 제공된 추천 후보와 거래데이터 신호만 근거로 답변한다.
- 데이터에 없는 납기, 재고, 인쇄 가능 여부는 단정하지 않는다.
- 가격과 최소구매수량은 제공된 값만 말한다.
- 고객에게 보여줄 수 있게 자연스러운 한국어로 답변한다.
- 상위 3~5개 상품을 추천하고, 각 상품의 추천 이유를 짧게 적는다.
- 마지막에 확인이 필요한 조건을 적는다.
"""

    user_prompt = f"""
아래 근거를 바탕으로 고객에게 상품 추천 답변을 작성해줘.

{context}
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_prompt.strip()},
        ],
        options={
            "temperature": 0.2,
            "num_ctx": 4096,
        },
        stream=False,
    )

    return {
        "answer": response["message"]["content"],
        "result": result,
    }


# 사용 예시
# answer_result = generate_customer_answer_with_ollama(
#     "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
#     model=LLM_MODEL,
#     top_k=5,
# )
# print(answer_result["answer"])
# display(answer_result["result"]["recommendations"])


## 15. 거래데이터 profile 단위 임베딩 실험 (patch)

위 9번 '유사 거래 사례 검색'은 거래데이터를 row 단위 그대로 임베딩합니다.
이 섹션은 같은 문제를 다른 방식으로 풀어본 실험입니다: 구매처분류 x 상품분류 x 주문월 x 행사/대상/시즌 단위로 거래 row를 먼저 profile로 묶은 뒤, 그 profile을 임베딩합니다.

원래는 별도 노트북(`product_recommendation_v1_profile_patch.ipynb`)으로 있었는데, 기록을 한 곳에 모으기 위해 이 노트북에 합쳤습니다.

**두 방식은 서로 대체 관계가 아니라 별도 함수로 공존합니다.** row 단위 함수(`retrieve_similar_trades`, `extract_trade_signals`, `recommend_products_v1`)는 그대로 남아있고,
아래 profile 단위 함수(`retrieve_similar_trade_profiles`, `extract_profile_signals`, `recommend_products_v1_profile`)를 추가로 씁니다. 필요할 때 둘 다 호출해서 결과를 비교할 수 있습니다.

## 1. 거래 Profile 생성

거래데이터를 개별 row가 아니라 패턴 단위로 묶습니다.

묶는 기준:

```text
구매처 분류
+ 상품분류
+ 주문월
+ 행사필터
+ 대상필터
+ 시즌필터
```

`구매처명`과 `거래 상품명`은 group 기준에 넣지 않고, 대표 샘플로만 보관합니다.  
이렇게 해야 profile 수가 지나치게 늘어나지 않습니다.


In [ ]:
# =========================
# 1. 거래 Profile 생성
# =========================

def join_top_values(series, n=8):
    values = [clean_text(x) for x in series.tolist() if clean_text(x)]
    values = list(dict.fromkeys(values))
    return " / ".join(values[:n])


def make_trade_profiles(trade_df: pd.DataFrame) -> pd.DataFrame:
    """
    거래 row를 추천용 profile 단위로 집계합니다.

    핵심:
    - 구매처명은 group 기준에서 제외
    - 상품명도 group 기준에서 제외
    - 대신 대표 구매처명/대표 상품명으로 샘플 보관
    """

    group_cols = [
        "buyer_type_large",
        "buyer_type_middle",
        "buyer_type_small",
        "buyer_type_detail",
        "trade_category_large",
        "trade_category_middle",
        "trade_category_small",
        "order_month",
        "event_filter",
        "target_filter",
        "season_filter",
    ]

    group_cols = [c for c in group_cols if c in trade_df.columns]

    profile_df = (
        trade_df
        .groupby(group_cols, dropna=False)
        .agg(
            trade_count=("trade_product_name", "count"),
            sample_products=("trade_product_name", lambda x: join_top_values(x, n=10)),
            sample_buyer_names=("buyer_name", lambda x: join_top_values(x, n=8)),
            sample_print_methods=("print_method", lambda x: join_top_values(x, n=5)),
            first_order_date=("order_date", "min"),
            last_order_date=("order_date", "max"),
            median_bulk_price=("bulk_price", "median"),
            median_middle_price=("middle_price", "median"),
            median_small_price=("small_price", "median"),
            median_moq=("trade_moq", "median"),
            max_bulk_price=("bulk_price", "max"),
            max_middle_price=("middle_price", "max"),
            max_small_price=("small_price", "max"),
        )
        .reset_index()
    )

    profile_df["buyer_type_path"] = (
        profile_df["buyer_type_large"].fillna("").astype(str) + " > " +
        profile_df["buyer_type_middle"].fillna("").astype(str) + " > " +
        profile_df["buyer_type_small"].fillna("").astype(str) + " > " +
        profile_df["buyer_type_detail"].fillna("").astype(str)
    ).map(clean_text)

    profile_df["trade_category_path"] = (
        profile_df["trade_category_large"].fillna("").astype(str) + " > " +
        profile_df["trade_category_middle"].fillna("").astype(str) + " > " +
        profile_df["trade_category_small"].fillna("").astype(str)
    ).map(clean_text)

    profile_df["trade_profile_id"] = [
        f"TP-{i+1:06d}" for i in range(len(profile_df))
    ]

    profile_df["trade_profile_search_text"] = (
        "[구매처분류] " + profile_df["buyer_type_path"] + "\n" +
        "[주문월] " + profile_df["order_month"].fillna("").astype(str) + "월\n" +
        "[상품분류] " + profile_df["trade_category_path"] + "\n" +
        "[대표상품] " + profile_df["sample_products"].fillna("").astype(str) + "\n" +
        "[대표구매처] " + profile_df["sample_buyer_names"].fillna("").astype(str) + "\n" +
        "[행사/대상/시즌] " +
            profile_df["event_filter"].fillna("").astype(str) + " " +
            profile_df["target_filter"].fillna("").astype(str) + " " +
            profile_df["season_filter"].fillna("").astype(str) + "\n" +
        "[인쇄방법] " + profile_df["sample_print_methods"].fillna("").astype(str) + "\n" +
        "[거래수] " + profile_df["trade_count"].astype(str)
    ).map(clean_text)

    # 거래량 점수: 거래가 많이 쌓인 profile을 조금 더 중요하게 보기 위한 점수
    profile_df["trade_count_weight"] = np.log1p(profile_df["trade_count"])
    max_w = profile_df["trade_count_weight"].max()

    if max_w and max_w > 0:
        profile_df["trade_count_score"] = profile_df["trade_count_weight"] / max_w
    else:
        profile_df["trade_count_score"] = 0.0

    return profile_df


trade_profile_df = make_trade_profiles(trade_df)

print("원본 거래 row 수:", len(trade_df))
print("거래 profile 수:", len(trade_profile_df))
print("감소율:", f"{(1 - len(trade_profile_df) / max(len(trade_df), 1)) * 100:.1f}%")

PROFILE_PATH = OUTPUT_DIR / "reco_v1_profile_trade_profile_table.xlsx"
trade_profile_df.to_excel(PROFILE_PATH, index=False)
print("거래 profile 저장:", PROFILE_PATH)

display(trade_profile_df[[
    "trade_profile_id",
    "trade_count",
    "order_month",
    "buyer_type_path",
    "trade_category_path",
    "sample_products",
    "event_filter",
    "target_filter",
    "season_filter",
    "trade_count_score"
]].head(20))


## 2. 거래 Profile 인덱스 생성

기존 v1의 아래 코드는 사용하지 않습니다.

```python
trade_index = build_embedding_or_tfidf(
    name="trade",
    texts=trade_df["trade_search_text"].tolist(),
    ...
)
```

대신 아래처럼 `trade_profile_df`를 임베딩합니다.


In [ ]:
# =========================
# 2. 상품 인덱스 + 거래 Profile 인덱스 생성
# =========================

# 상품은 기존과 동일하게 상품 row 단위 임베딩
product_index = build_embedding_or_tfidf(
    name="product",
    texts=product_df["product_search_text"].tolist(),
    use_ollama=USE_OLLAMA_EMBEDDING,
    model=EMBED_MODEL,
)

# 거래는 row가 아니라 profile 단위 임베딩
trade_profile_index = build_embedding_or_tfidf(
    name="trade_profile",
    texts=trade_profile_df["trade_profile_search_text"].tolist(),
    use_ollama=USE_OLLAMA_EMBEDDING,
    model=EMBED_MODEL,
)

print("product backend:", product_index["backend"])
print("trade_profile backend:", trade_profile_index["backend"])
print("상품 row 수:", len(product_df))
print("거래 row 수:", len(trade_df))
print("거래 profile 수:", len(trade_profile_df))


## 3. 유사 거래 Profile 검색 함수

기존 `retrieve_similar_trades()` 대신 사용합니다.

점수 구조:

```text
profile_total_score =
profile_semantic_score * 0.50
+ date_lead_score * 0.20
+ profile_condition_score * 0.15
+ trade_count_score * 0.15
```

`trade_count_score`가 추가되어, 실제 거래가 많이 누적된 profile을 약간 더 중요하게 봅니다.


In [ ]:
# =========================
# 3. 유사 거래 Profile 검색
# =========================

def build_trade_profile_query_text(profile: dict) -> str:
    parts = [
        profile.get("raw_query", ""),
        profile.get("buyer_context", ""),
        profile.get("event_context", ""),
        profile.get("purpose", ""),
        profile.get("season", ""),
        " ".join(profile.get("style_keywords") or []),
        " ".join(profile.get("must_have") or []),
    ]
    return combine_text(*parts)


def calc_profile_condition_score(row, profile: dict) -> float:
    score = 0.0

    budget = profile.get("budget_max")
    quantity = profile.get("quantity")
    season = profile.get("season")

    # 예산과 profile 가격대 유사성
    if budget not in [None, ""] and not pd.isna(budget):
        try:
            budget = float(budget)
            prices = [
                row.get("median_bulk_price", np.nan),
                row.get("median_middle_price", np.nan),
                row.get("median_small_price", np.nan),
            ]
            valid_prices = [p for p in prices if not pd.isna(p)]

            if valid_prices:
                min_price = min(valid_prices)
                if min_price <= budget:
                    score += 0.35
                elif min_price <= budget * 1.2:
                    score += 0.15
        except Exception:
            pass

    # 수량과 최소구매수량
    if quantity not in [None, ""] and not pd.isna(quantity):
        try:
            quantity = float(quantity)
            moq = row.get("median_moq", np.nan)
            if pd.isna(moq) or moq <= quantity:
                score += 0.25
        except Exception:
            pass

    # 시즌 필터가 있는 경우 보조 점수
    if season and season in str(row.get("season_filter", "")):
        score += 0.20

    return min(score, 1.0)


def retrieve_similar_trade_profiles(query: str, top_k: int = 20, profile: dict | None = None) -> pd.DataFrame:
    if profile is None:
        profile = extract_query_profile_with_ollama(query)

    query_text = build_trade_profile_query_text(profile)
    semantic_scores = search_index(query_text, trade_profile_index)

    result = trade_profile_df.copy()
    result["profile_semantic_score"] = semantic_scores
    result["date_lead_score"] = result["order_month"].apply(lambda m: calc_date_lead_score(m, profile))
    result["profile_condition_score"] = result.apply(lambda row: calc_profile_condition_score(row, profile), axis=1)

    result["profile_total_score"] = (
        result["profile_semantic_score"] * 0.50 +
        result["date_lead_score"] * 0.20 +
        result["profile_condition_score"] * 0.15 +
        result["trade_count_score"] * 0.15
    )

    return (
        result
        .sort_values("profile_total_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"
test_profile = extract_query_profile_with_ollama(test_query)

print("[추출 프로필]")
print(json.dumps(test_profile, ensure_ascii=False, indent=2))

print("\n[참고 주문월 가중치]")
print(get_reference_month_weights_from_profile(test_profile))

similar_profiles = retrieve_similar_trade_profiles(test_query, top_k=10, profile=test_profile)

display(similar_profiles[[
    "trade_profile_id",
    "profile_total_score",
    "profile_semantic_score",
    "date_lead_score",
    "profile_condition_score",
    "trade_count_score",
    "trade_count",
    "order_month",
    "buyer_type_path",
    "trade_category_path",
    "sample_products",
    "event_filter",
    "target_filter",
    "season_filter"
]])


## 4. 거래 Profile 신호 추출

기존 `extract_trade_signals()` 대신 사용합니다.

유사 거래 profile에서 아래 신호를 추출합니다.

```text
자주 등장한 상품분류
대표 상품명
구매처 유형
행사/대상/시즌
가격대
```


In [ ]:
# =========================
# 4. 거래 Profile 신호 추출
# =========================

def weighted_value_counts_from_profiles(df, col, weight_col="profile_total_score", top_n=10):
    rows = []

    for _, row in df.iterrows():
        value = clean_text(row.get(col, ""))
        if not value:
            continue

        weight = float(row.get(weight_col, 0))
        trade_count = int(row.get("trade_count", 1))

        rows.append((value, weight, trade_count))

    if not rows:
        return pd.DataFrame(columns=[col, "weighted_score", "profile_count", "trade_count_sum", "signal_score"])

    temp = pd.DataFrame(rows, columns=[col, "weight", "trade_count"])

    out = (
        temp.groupby(col)
        .agg(
            weighted_score=("weight", "sum"),
            profile_count=("weight", "count"),
            trade_count_sum=("trade_count", "sum"),
        )
        .reset_index()
    )

    # 유사도 점수와 실제 거래량을 함께 반영
    out["signal_score"] = out["weighted_score"] * np.log1p(out["trade_count_sum"])

    return (
        out.sort_values("signal_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


def extract_profile_signals(similar_profiles: pd.DataFrame) -> dict:
    category_large = weighted_value_counts_from_profiles(similar_profiles, "trade_category_large", top_n=10)
    category_middle = weighted_value_counts_from_profiles(similar_profiles, "trade_category_middle", top_n=10)
    category_small = weighted_value_counts_from_profiles(similar_profiles, "trade_category_small", top_n=10)
    buyers = weighted_value_counts_from_profiles(similar_profiles, "buyer_type_middle", top_n=10)
    events = weighted_value_counts_from_profiles(similar_profiles, "event_filter", top_n=10)
    targets = weighted_value_counts_from_profiles(similar_profiles, "target_filter", top_n=10)
    seasons = weighted_value_counts_from_profiles(similar_profiles, "season_filter", top_n=10)

    # 대표 상품명은 sample_products에서 다시 분해해서 수집
    product_rows = []

    for _, row in similar_profiles.iterrows():
        samples = clean_text(row.get("sample_products", ""))
        if not samples:
            continue

        for item in samples.split("/"):
            item = clean_text(item)
            if not item:
                continue

            product_rows.append({
                "sample_product": item,
                "weight": float(row.get("profile_total_score", 0)),
                "trade_count": int(row.get("trade_count", 1)),
            })

    if product_rows:
        temp_products = pd.DataFrame(product_rows)
        products = (
            temp_products
            .groupby("sample_product")
            .agg(
                weighted_score=("weight", "sum"),
                count=("weight", "count"),
                trade_count_sum=("trade_count", "sum"),
            )
            .reset_index()
        )
        products["signal_score"] = products["weighted_score"] * np.log1p(products["trade_count_sum"])
        products = products.sort_values("signal_score", ascending=False).head(10).reset_index(drop=True)
    else:
        products = pd.DataFrame(columns=["sample_product", "weighted_score", "count", "trade_count_sum", "signal_score"])

    price_cols = ["median_bulk_price", "median_middle_price", "median_small_price"]
    prices = []

    for c in price_cols:
        if c in similar_profiles.columns:
            prices.extend([x for x in similar_profiles[c].tolist() if not pd.isna(x)])

    price_stats = {}
    if prices:
        arr = np.array(prices, dtype=float)
        price_stats = {
            "min": float(np.nanmin(arr)),
            "q25": float(np.nanpercentile(arr, 25)),
            "median": float(np.nanmedian(arr)),
            "q75": float(np.nanpercentile(arr, 75)),
            "max": float(np.nanmax(arr)),
        }

    signal_parts = []

    def add_top_values(title, df, col):
        vals = df[col].head(8).tolist() if len(df) else []
        vals = [clean_text(v) for v in vals if clean_text(v)]
        if vals:
            signal_parts.append(f"[{title}] " + " ".join(vals))

    add_top_values("유사거래 상품분류 대", category_large, "trade_category_large")
    add_top_values("유사거래 상품분류 중", category_middle, "trade_category_middle")
    add_top_values("유사거래 상품분류 소", category_small, "trade_category_small")
    add_top_values("유사거래 대표상품", products, "sample_product")
    add_top_values("유사구매처", buyers, "buyer_type_middle")
    add_top_values("행사", events, "event_filter")
    add_top_values("대상", targets, "target_filter")
    add_top_values("시즌", seasons, "season_filter")

    signal_text = clean_text("\n".join(signal_parts))

    return {
        "category_large": category_large,
        "category_middle": category_middle,
        "category_small": category_small,
        "products": products,
        "buyers": buyers,
        "events": events,
        "targets": targets,
        "seasons": seasons,
        "price_stats": price_stats,
        "signal_text": signal_text,
    }


signals = extract_profile_signals(similar_profiles)

print("[거래 Profile 신호 텍스트]")
print(signals["signal_text"])

print("\n[가격대 통계]")
print(signals["price_stats"])

print("\n[상위 상품분류 중]")
display(signals["category_middle"])

print("\n[상위 대표상품]")
display(signals["products"])


## 5. 상품 랭킹 함수 수정

기존 `calc_trade_category_signal_score()`와 `rank_products()`를 아래 코드로 교체합니다.

profile 방식에서는 거래 row가 아니라 거래 profile 신호를 기준으로 상품을 재랭킹합니다.


In [ ]:
# =========================
# 5. 상품 랭킹 함수 수정
# =========================

def calc_profile_category_signal_score(row, signals: dict):
    product_text = combine_text(
        row.get("category_path", ""),
        row.get("product_name", ""),
        row.get("keywords", ""),
    )

    score = 0.0

    for signal_key, col_name, weight in [
        ("category_middle", "trade_category_middle", 0.55),
        ("category_small", "trade_category_small", 0.35),
        ("category_large", "trade_category_large", 0.10),
    ]:
        signal_df = signals.get(signal_key)

        if signal_df is None or len(signal_df) == 0:
            continue

        max_signal = signal_df["signal_score"].max() if "signal_score" in signal_df.columns else signal_df["weighted_score"].max()

        for _, srow in signal_df.iterrows():
            value = clean_text(srow.get(col_name, ""))
            if not value:
                continue

            if value in product_text:
                raw_score = srow.get("signal_score", srow.get("weighted_score", 0))
                normalized = float(raw_score) / max_signal if max_signal else 0
                score += normalized * weight

    return min(score, 1.0)


def rank_products_profile(query: str, profile: dict, signals: dict, top_k: int = 5) -> pd.DataFrame:
    product_query_text = build_product_query_text(profile)

    query_product_scores = search_index(product_query_text, product_index)

    signal_text = signals.get("signal_text", "")
    if signal_text:
        trade_signal_product_scores = search_index(signal_text, product_index)
    else:
        trade_signal_product_scores = np.zeros(len(product_df))

    result = product_df.copy()

    result["query_product_score"] = query_product_scores
    result["trade_signal_product_score"] = trade_signal_product_scores

    result["profile_category_signal_score"] = result.apply(
        lambda row: calc_profile_category_signal_score(row, signals),
        axis=1,
    )

    budget = profile.get("budget_max")
    quantity = profile.get("quantity")

    result["budget_score"] = result["price"].apply(lambda p: calc_budget_score(p, budget))
    result["quantity_score"] = result["moq"].apply(lambda q: calc_quantity_score(q, quantity))
    result["product_quality_score"] = result.apply(calc_product_quality_score, axis=1)

    # profile 방식에서는 거래 신호 비중을 높게 둠
    result["final_score"] = (
        result["query_product_score"] * 0.18 +
        result["trade_signal_product_score"] * 0.32 +
        result["profile_category_signal_score"] * 0.22 +
        result["budget_score"] * 0.15 +
        result["quantity_score"] * 0.05 +
        result["product_quality_score"] * 0.08
    )

    def make_reason(row):
        reasons = []

        if row["trade_signal_product_score"] >= 0.55:
            reasons.append("거래 Profile 신호와 의미적으로 가까움")
        elif row["trade_signal_product_score"] >= 0.35:
            reasons.append("거래 Profile 신호와 일부 관련 있음")

        if row["profile_category_signal_score"] >= 0.5:
            reasons.append("유사 거래 Profile에서 나온 상품분류와 현재 상품분류가 일치")
        elif row["profile_category_signal_score"] > 0:
            reasons.append("유사 거래 Profile의 상품분류와 일부 겹침")

        if row["query_product_score"] >= 0.55:
            reasons.append("사용자 요청과 상품 설명의 의미 유사도 높음")
        elif row["query_product_score"] >= 0.35:
            reasons.append("사용자 요청과 상품 설명이 일부 유사함")

        if budget not in [None, ""] and not pd.isna(budget):
            if pd.notna(row["price"]) and float(row["price"]) <= float(budget):
                reasons.append(f"예산 {int(float(budget)):,}원 이하 조건 충족")
            elif pd.notna(row["price"]):
                reasons.append("예산 초과 여부 확인 필요")
            else:
                reasons.append("가격 정보 확인 필요")

        if quantity not in [None, ""] and not pd.isna(quantity):
            if pd.isna(row["moq"]) or float(row["moq"]) <= float(quantity):
                reasons.append("요청 수량 기준 최소구매수량 충족 가능")
            else:
                reasons.append("최소구매수량 확인 필요")

        if not reasons:
            reasons.append("현재 상품데이터 기준 추천 후보로 검토 가능")

        return " / ".join(reasons)

    result["recommend_reason"] = result.apply(make_reason, axis=1)

    output_cols = [
        "product_id",
        "product_name",
        "price",
        "moq",
        "category_path",
        "keywords",
        "query_product_score",
        "trade_signal_product_score",
        "profile_category_signal_score",
        "budget_score",
        "quantity_score",
        "product_quality_score",
        "final_score",
        "recommend_reason",
        "image",
    ]

    return (
        result
        .sort_values("final_score", ascending=False)
        .head(top_k)
        [output_cols]
        .reset_index(drop=True)
    )


## 6. 최종 추천 함수 교체

기존 `recommend_products_v1()` 대신 아래 함수를 사용합니다.


In [ ]:
# =========================
# 6. 최종 추천 함수
# =========================

def recommend_products_v1_profile(query: str, top_k: int = DEFAULT_TOP_K, profile_top_k: int = 30):
    profile = extract_query_profile_with_ollama(query)

    similar_profiles = retrieve_similar_trade_profiles(
        query=query,
        top_k=profile_top_k,
        profile=profile,
    )

    signals = extract_profile_signals(similar_profiles)

    recommendations = rank_products_profile(
        query=query,
        profile=profile,
        signals=signals,
        top_k=top_k,
    )

    return {
        "query": query,
        "profile": profile,
        "reference_month_weights": get_reference_month_weights_from_profile(profile),
        "similar_profiles": similar_profiles,
        "signals": signals,
        "recommendations": recommendations,
        "backend": {
            "product": product_index["backend"],
            "trade_profile": trade_profile_index["backend"],
            "embedding_model": product_index.get("model"),
        },
        "counts": {
            "trade_rows": len(trade_df),
            "trade_profiles": len(trade_profile_df),
            "products": len(product_df),
        }
    }


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"

result = recommend_products_v1_profile(test_query, top_k=5, profile_top_k=20)

print("[질문]")
print(result["query"])

print("\n[추출 프로필]")
print(json.dumps(result["profile"], ensure_ascii=False, indent=2))

print("\n[참고 주문월 가중치]")
print(result["reference_month_weights"])

print("\n[사용 backend]")
print(result["backend"])

print("\n[데이터 수]")
print(result["counts"])

print("\n[거래 Profile 신호 텍스트]")
print(result["signals"]["signal_text"])

print("\n[추천 상품]")
display(result["recommendations"])

print("\n[유사 거래 Profile]")
display(result["similar_profiles"][[
    "trade_profile_id",
    "profile_total_score",
    "profile_semantic_score",
    "date_lead_score",
    "profile_condition_score",
    "trade_count_score",
    "trade_count",
    "order_month",
    "buyer_type_path",
    "trade_category_path",
    "sample_products",
    "event_filter",
    "target_filter",
    "season_filter"
]].head(10))


## 7. 여러 질문 테스트 및 저장

In [ ]:
test_queries = [
    "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
    "병원 개원 답례품으로 3천원 이하 500개 추천해줘",
    "대학교 OT에서 나눠줄 저렴한 사은품 추천해줘",
    "회사 창립기념품으로 실용적인 상품 추천해줘",
    "박람회 부스에서 나눠줄 홍보물 추천해줘",
    "12월 연말 고객 선물로 5만원 이하 상품 추천해줘",
    "어린이집 행사 답례품 추천해줘",
    "로고 인쇄 가능한 텀블러 추천해줘",
]

all_rows = []

for query in test_queries:
    result = recommend_products_v1_profile(query, top_k=5, profile_top_k=20)

    profile = result["profile"]
    ref_months = result["reference_month_weights"]

    top_trade_categories = []
    if len(result["signals"]["category_middle"]):
        top_trade_categories = result["signals"]["category_middle"]["trade_category_middle"].head(5).tolist()

    for rank, (_, row) in enumerate(result["recommendations"].iterrows(), start=1):
        all_rows.append({
            "query": query,
            "rank": rank,
            "buyer_context": profile.get("buyer_context", ""),
            "event_context": profile.get("event_context", ""),
            "purpose": profile.get("purpose", ""),
            "season": profile.get("season", ""),
            "event_month": profile.get("event_month", ""),
            "reference_month_weights": json.dumps(ref_months, ensure_ascii=False),
            "top_trade_categories": ", ".join(top_trade_categories),
            "product_id": row["product_id"],
            "product_name": row["product_name"],
            "price": row["price"],
            "moq": row["moq"],
            "category_path": row["category_path"],
            "query_product_score": row["query_product_score"],
            "trade_signal_product_score": row["trade_signal_product_score"],
            "profile_category_signal_score": row["profile_category_signal_score"],
            "budget_score": row["budget_score"],
            "quantity_score": row["quantity_score"],
            "product_quality_score": row["product_quality_score"],
            "final_score": row["final_score"],
            "recommend_reason": row["recommend_reason"],
            "trade_rows": result["counts"]["trade_rows"],
            "trade_profiles": result["counts"]["trade_profiles"],
        })

batch_result_df = pd.DataFrame(all_rows)

BATCH_RESULT_PATH = OUTPUT_DIR / "reco_v1_profile_batchtest.xlsx"
batch_result_df.to_excel(BATCH_RESULT_PATH, index=False)

print("추천 결과 저장:", BATCH_RESULT_PATH)
display(batch_result_df.head(30))


## 16. v1에서 추가로 보완하면 좋은 점

이번 v1은 구조적으로 v0보다 좋아졌지만, 실제 운영용으로 가려면 아래를 보완하면 좋습니다.

---

### 1. 거래데이터 날짜의 의미 분리

현재 `날짜`를 주문일자로 가정했습니다.

하지만 가능하면 앞으로는 아래를 분리해서 저장하는 것이 좋습니다.

```text
주문일자
출고요청일
납품일
행사/사용 예정일
```

지금은 고객 질문의 “8월 행사”를 사용월로 보고, 거래데이터의 `날짜`를 주문월로 보정하고 있습니다.  
나중에 `행사/사용 예정일`이 실제로 쌓이면 추천 정확도가 훨씬 좋아집니다.

---

### 2. 상품별 제작 리드타임 반영

현재는 기본값으로 1~2개월 앞선 주문월을 참고합니다.

하지만 상품마다 다릅니다.

```text
기성품/빠른 납품 상품: 1~2주
인쇄 상품: 2~4주
OEM/ODM 상품: 1~3개월 이상
```

상품데이터에 `납기`, `제작기간`, `인쇄기간` 같은 컬럼이 있으면 추천 품질이 좋아집니다.

---

### 3. 가격 구조 개선

현재 상품데이터의 `상품판매가`를 주로 사용합니다.

하지만 판촉물은 수량별 단가가 중요합니다.

```text
50개 단가
100개 단가
300개 단가
500개 단가
1000개 단가
```

상품데이터의 `복수구매` 컬럼 안에 수량별 가격이 들어가 있으므로,  
v2에서는 사용자 수량에 맞는 단가를 파싱해서 가격 점수를 계산하는 것이 좋습니다.

---

### 4. 추천 제외 조건

운영용에서는 아래 조건을 반영해야 합니다.

```text
판매중지 상품 제외
회원전용 상품 제외 또는 별도 표시
재고 없음 제외
이미지 없음 상품 후순위
가격 없음 상품 후순위
```

현재 v1은 샘플 테스트용이라 완전히 제외하지 않고 점수로만 약하게 반영합니다.

---

### 5. 거래데이터 가중치 개선

현재는 유사 거래 사례의 점수를 기준으로 상품분류 신호를 추출합니다.

추후에는 아래를 추가하면 좋습니다.

```text
최근 거래 가중치
주문금액 가중치
수량 가중치
반복 구매처 가중치
동일 업종 가중치
```

예를 들어 최근 2년 거래를 더 중요하게 보거나,  
대량 주문이 많았던 상품군을 더 높게 볼 수 있습니다.

---

### 6. 추천 평가셋 구축

상품 추천은 정답이 1개가 아닙니다.  
따라서 FAQ처럼 `expected_faq_id` 방식보다는 아래 평가가 맞습니다.

```text
예산 조건 충족 여부
수량 조건 충족 여부
구매처/행사 맥락 적합성
상품군 다양성
실제 판매 가능성
추천 이유의 납득성
담당자 수동 평가 점수
```

평가 파일 예시:

```text
query
추천상품1
추천상품2
추천상품3
예산적합
수량적합
맥락적합
담당자평점
비고
```

---

### 7. 대용량 적용 시 Vector DB 도입

샘플 100개에서는 메모리에서 바로 임베딩 검색해도 됩니다.

하지만 상품 4만 개 이상, 거래데이터 10만 건 이상이면 아래 구조가 좋습니다.

```text
Qdrant 또는 Chroma
상품 컬렉션
거래 컬렉션
metadata filter
hybrid search
reranking
```

---

### 8. Reranker 추가

임베딩 검색은 후보를 넓게 찾는 데 좋지만, 최종 순위는 애매할 수 있습니다.

v2 이후에는:

```text
1차: 임베딩 검색으로 후보 100개
2차: 조건 필터
3차: reranker 또는 LLM scoring으로 Top 10 정렬
```

구조가 좋습니다.

---

### 9. 상담/주문 데이터 추가

상품 추천에는 상품명보다 상담 내용이 더 중요한 경우가 많습니다.

앞으로 상담 로그에서 아래를 구조화하면 좋습니다.

```text
고객 요청 문장
추천한 상품
실제 주문 여부
견적 금액
탈락 사유
담당자 메모
```

이 데이터가 쌓이면 “추천 → 실제 구매” 기반으로 고도화할 수 있습니다.

---

## v1 한 줄 요약

```text
수동 사전 없이,
사용자 질문을 구조화하고,
거래데이터에서 비슷한 구매 맥락과 시기 신호를 찾고,
현재 상품데이터를 임베딩과 조건 점수로 재랭킹하는 구조입니다.
```
